In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import xarray as xr
from pystac_client import Client
from odc.stac import load, configure_s3_access
from odc.geo import BoundingBox

In [ ]:
# Set up the query
catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

In [ ]:
get_scene = 'S1A_IW_SLC__1SSH_20221103T233154_20221103T233222_045736_05785B_8063'
roi_bbox = BoundingBox(
        left=-27.8,
        bottom=-75.8,
        right=-26,
        top=-75.4,
        crs="EPSG:4326"
    )
roi_bbox.explore()

In [ ]:
# Query by sceneID
scene_filter = {
    "op": "=",
    "args": [{"property": "sarard:scene_id"}, get_scene]
}

scene_items = stac_client.search(
    collections=["ga_s1_nrb_iw_vv_vh_0","ga_s1_nrb_iw_hh_hv_0",'ga_s1_nrb_iw_hh_0'],
    bbox = roi_bbox.bbox,
    filter=scene_filter,
).item_collection()

print(f"Found {len(scene_items)} items")

In [ ]:
scene_items


In [ ]:
# Assets to load
assets_to_load = ["HH_gamma0","number_of_looks"]

# CRS and resolution
output_crs = "epsg:3031"
output_res = 20 # 20m is the native resolution of the data. Here, we resample to 400m to save memory

# Property or function to group by. "solar_day" is already built into odc-stac
groupy_by_operation = "solar_day"

In [ ]:
# Load the data
ds = load(
    items=scene_items,
    bands=assets_to_load,
    intersects = roi_bbox.boundary(),
    crs=output_crs,
    resolution=output_res,
    groupby=groupy_by_operation,
    chunks={},
)

In [ ]:
ds.compute()

In [ ]:
# Quickly plot to visualise the data
plt.figure(figsize=(12, 12), dpi=300)
ds["HH_gamma0_dB"] = 10*np.log10(ds["HH_gamma0"])
ds["HH_gamma0_dB"].plot(vmin =-10, vmax = 0, cmap="Greys_r")
plt.show()

In [ ]:
import sys
sys.path.insert(1, './src/')
from speckle_filters import apply_lee_filter

ds["HH_lee_filtered"] = apply_lee_filter(ds["HH_gamma0"], size=5)
ds["HH_lee_filtered_dB"] = 10*np.log10(ds["HH_lee_filtered"])
plt.figure(figsize=(12, 12), dpi=300)
ds["HH_lee_filtered_dB"].plot(vmin =-10, vmax = 0, cmap="Greys_r")
plt.show()

In [ ]:
import numpy as np
import rasterio
import sys
sys.path.insert(1,'./src/')
from lee_sigma_improved_xr import lee_sigma_improved_xr  # import the function we wrote

In [ ]:
lee_improved_adapt_filtered = lee_sigma_improved_xr(
    ds, 
    'HH_gamma0',
    'number_of_looks', 
    data_type = 'intensity',
    win=(9, 9)
)

In [ ]:
lee_improved_adapt_filtered_db = 10*np.log10(lee_improved_adapt_filtered)
plt.figure(figsize=(12, 12), dpi=300) 
lee_improved_adapt_filtered_db.plot.imshow(cmap="gray", vmin =-10, vmax = 0)
plt.show()